# Airline Departure Delay Prediction: Data Loading
**Team:** Sam Wagstaff, Mason Lo, Prathik Nagulugari

Loads and prepares the primary source from our sources activity: the **Bureau of Transportation Statistics (BTS)** On-Time Performance data (https://transtats.bts.gov/).

**Target:** `DELAYED` = departure delay **> 15 min** (BTS `DepDel15`; the definition used in the literature, e.g. Esmaeilzadeh & Mokhtarimousavi, 2020).

**How to get the data:** On transtats.bts.gov go to *Aviation Data Library → Airline On-Time Performance Data → Reporting Carrier On-Time Performance (1987-present) → Download*. Select the fields listed in `USECOLS` below, download one or more months, unzip, and put the CSVs in `data/`.

In [ ]:
import glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_GLOB = "data/*.csv"   # BTS monthly CSVs (zip files are fine too: data/*.zip)
DELAY_MIN = 15             # delayed = departure delay strictly greater than this

# Known at scheduled/planning time -> safe predictors
FEATURE_COLS = ["Month", "DayOfWeek", "DayofMonth", "Reporting_Airline", "Origin", "Dest",
                "CRSDepTime", "DepTimeBlk", "CRSArrTime", "CRSElapsedTime", "Distance"]
# Recorded only after the prediction would be made -> LEAKAGE, never use as features
LEAKAGE_COLS = ["DepTime", "DepDelay", "DepDelayMinutes", "DepDel15", "TaxiOut", "WheelsOff",
                "ArrTime", "ArrDelay", "ArrDel15", "CarrierDelay", "WeatherDelay",
                "NASDelay", "SecurityDelay", "LateAircraftDelay"]
KEEP_EXTRA = ["FlightDate", "Cancelled", "Diverted", "DepDelay", "DepDel15"]  # for target + QA
USECOLS = sorted(set(FEATURE_COLS + KEEP_EXTRA))

## 1. Load

In [ ]:
import os
import glob
import pandas as pd


DATA_GLOB = r"C:\Users\prath\Downloads\On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2026_1.zip"


files = sorted(glob.glob(DATA_GLOB))

if not files:
    raise FileNotFoundError(
        "No BTS files found. Check the ZIP file path."
    )

frames = []

for f in files:
    # Read only the required columns
    header = pd.read_csv(f, nrows=0).columns
    cols = [c for c in USECOLS if c in header]

    frames.append(
        pd.read_csv(f, usecols=cols, low_memory=False)
        .assign(source_file=os.path.basename(f))
    )

    print(f"{os.path.basename(f)}: {len(frames[-1]):,} rows")

raw = pd.concat(frames, ignore_index=True)

print(f"\nTotal: {raw.shape[0]:,} rows × {raw.shape[1]} cols")

raw.head()

ValueError: Multiple files found in ZIP file. Only one file per ZIP: ['On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_1.csv', 'readme.html']

## 2. Clean and define the target
Cancelled and diverted flights have no departure delay, so they are dropped.

In [ ]:
df = raw.copy()
n0 = len(df)
df = df[(df["Cancelled"] == 0) & (df["Diverted"] == 0)]
df = df.dropna(subset=["DepDelay"])
print(f"Dropped {n0-len(df):,} cancelled/diverted/missing-delay rows ({(n0-len(df))/n0:.1%})")

df["DELAYED"] = (df["DepDelay"] > DELAY_MIN).astype(int)
df["FlightDate"] = pd.to_datetime(df["FlightDate"])
df["DepHour"] = (df["CRSDepTime"] // 100).clip(upper=23)   # scheduled hour 0-23

# Cross-check against BTS's own flag
if "DepDel15" in df:
    print("Agreement with BTS DepDel15:", (df["DELAYED"] == df["DepDel15"]).mean().round(4))
df.shape

## 3. Quality checks

In [ ]:
print("Missing values in feature columns:")
miss = df[FEATURE_COLS].isna().mean().loc[lambda s: s > 0].round(4)
print(miss.to_string() if len(miss) else "none")
print("\nDate range:", df["FlightDate"].min().date(), "to", df["FlightDate"].max().date())
print("Airlines:", df["Reporting_Airline"].nunique(), "| Origins:", df["Origin"].nunique())
rate = df["DELAYED"].mean()
print(f"\nClass balance: {rate:.1%} delayed vs {1-rate:.1%} on time")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
df.groupby("DepHour")["DELAYED"].mean().plot(ax=ax[0], marker="o"); ax[0].set_title("Delay rate by scheduled hour")
df.groupby("DayOfWeek")["DELAYED"].mean().plot.bar(ax=ax[1]); ax[1].set_title("By day of week (1=Mon)")
df.groupby("Reporting_Airline")["DELAYED"].mean().sort_values().plot.barh(ax=ax[2]); ax[2].set_title("By airline")
for a in ax: a.set_ylabel("")
plt.tight_layout(); plt.show()

## 4. Modeling table
Only planning-time features are kept; leakage columns are removed and asserted absent.

In [ ]:
X = df[FEATURE_COLS + ["DepHour"]].copy()
y = df["DELAYED"]
assert not set(X.columns) & set(LEAKAGE_COLS), "Leakage column in features!"
for c in ["Reporting_Airline", "Origin", "Dest", "DepTimeBlk"]:
    X[c] = X[c].astype("category")
print(X.shape, "| baseline accuracy (always 'on time'):", round(1 - y.mean(), 3))
X.head()

In [ ]:
# Time-based split avoids training on the future (use instead of a random split)
cutoff = df["FlightDate"].quantile(0.8)
train, test = df["FlightDate"] <= cutoff, df["FlightDate"] > cutoff
X_train, X_test, y_train, y_test = X[train], X[test], y[train], y[test]
print(f"train {len(X_train):,} | test {len(X_test):,} | delay rate {y_train.mean():.1%} / {y_test.mean():.1%}")